# automata_extensions : exemples

Visite guidée de la bibliothèque, qui étend `automata-lib` 9.2.0 sans la modifier. Chaque méthode renvoie un **nouvel** automate (immuabilité).

In [1]:
from automata_extensions.fa.dfa import ExtendedDFA
from automata_extensions.fa.nfa import ExtendedNFA

## 1. Construire un DFA

Le DFA ci-dessous reconnaît `a+` ; l'état `3` est un puits.

In [2]:
dfa = ExtendedDFA(
    states={'1', '2', '3'},
    input_symbols={'a', 'b'},
    transitions={
        '1': {'a': '2', 'b': '3'},
        '2': {'a': '2', 'b': '3'},
        '3': {'a': '3', 'b': '3'},
    },
    initial_state='1',
    final_states={'2'},
)
print(dfa.accepts_input('aaa'), dfa.accepts_input('ab'), dfa.accepts_input(''))

True False False


## 2. Analyse : accessibilité, utilité, trim

In [3]:
print('accessibles   :', sorted(dfa.accessible_states()))
print('coaccessibles :', sorted(dfa.coaccessible_states()))
print('utiles        :', sorted(dfa.useful_states()))
print('est trim ?    :', dfa.is_trim())
print('infini ?      :', dfa.is_language_infinite())
print('vide ?        :', dfa.is_language_empty())
print('SCC           :', dfa.strongly_connected_components())

accessibles   : ['1', '2', '3']
coaccessibles : ['1', '2']
utiles        : ['1', '2']
est trim ?    : False
infini ?      : True
vide ?        : False
SCC           : [{'3'}, {'2'}, {'1'}]


## 3. Immuabilité

`trim()` renvoie un nouvel automate ; l'original n'est jamais modifié.

In [4]:
t = dfa.trim()
print('original :', sorted(dfa.states))
print('trim     :', sorted(t.states))
print('objets distincts :', t is not dfa)

original : ['1', '2', '3']
trim     : ['1', '2']
objets distincts : True


## 4. Opérations sur les langages

In [5]:
# d2 : nombre pair de 'a'
d2 = ExtendedDFA(
    states={'p', 'i'},
    input_symbols={'a', 'b'},
    transitions={'p': {'a': 'i', 'b': 'p'}, 'i': {'a': 'p', 'b': 'i'}},
    initial_state='p',
    final_states={'p'},
)
inter = dfa.intersection(d2)
union = dfa.union(d2)
comp = dfa.complement()
for w in ['', 'a', 'aa', 'aaa', 'ab']:
    print(repr(w).ljust(6), 'dfa:', dfa.accepts_input(w),
          '| inter:', inter.accepts_input(w),
          '| union:', union.accepts_input(w),
          '| complement:', comp.accepts_input(w))

''     dfa: False | inter: False | union: True | complement: True
'a'    dfa: True | inter: False | union: True | complement: False
'aa'   dfa: True | inter: True | union: True | complement: False
'aaa'  dfa: True | inter: False | union: True | complement: False
'ab'   dfa: False | inter: False | union: False | complement: True


## 5. Minimisation : méthode classique contre Hopcroft

Ce DFA à 4 états reconnaît « nombre pair de `a` » : il se réduit à 2 états.

In [6]:
redondant = ExtendedDFA(
    states={'A', 'B', 'C', 'D'},
    input_symbols={'a', 'b'},
    transitions={
        'A': {'a': 'B', 'b': 'C'},
        'B': {'a': 'C', 'b': 'D'},
        'C': {'a': 'D', 'b': 'A'},
        'D': {'a': 'A', 'b': 'B'},
    },
    initial_state='A',
    final_states={'A', 'C'},
)
m1 = redondant.minimize()
m2 = redondant.minimize_hopcroft()
print('états avant      :', len(redondant.states))
print('minimize         :', len(m1.states))
print('minimize_hopcroft:', len(m2.states))
print('isomorphes       :', m1.is_isomorphic(m2))
print('même langage que d2 :', m1.language_equals(d2))

états avant      : 4
minimize         : 2
minimize_hopcroft: 2
isomorphes       : True
même langage que d2 : True


## 6. Expressions régulières

In [7]:
print('regex de dfa.trim()     :', dfa.trim().to_regex_arden())
print('regex de d2 (Arden)    :', d2.to_regex_arden())
print('regex de d2 (élimination d\'états) :', d2.to_regex_state_elimination())
nfa = ExtendedNFA.regex_to_nfa('a*b', input_symbols={'a', 'b'})
print('a*b accepte aab :', nfa.accepts_input('aab'))
print('dérivée de ab par a   :', ExtendedDFA.brzozowski_derivative('ab', 'a'))
print('dérivée de a*b par a  :', ExtendedDFA.brzozowski_derivative('a*b', 'a'))

regex de dfa.trim()     : aa*
regex de d2 (Arden)    : ((ab*a|b))*
regex de d2 (élimination d'états) : (ab*a|b)*
a*b accepte aab : True
dérivée de ab par a   : b
dérivée de a*b par a  : a*b


## 7. NFA avec transitions epsilon

Le symbole epsilon est `''`.

In [8]:
nfa_eps = ExtendedNFA(
    states={'q0', 'q1', 'q2'},
    input_symbols={'a', 'b'},
    transitions={
        'q0': {'a': {'q0'}, '': {'q1'}},
        'q1': {'b': {'q2'}},
        'q2': {},
    },
    initial_state='q0',
    final_states={'q2'},
)
sans_eps = nfa_eps.remove_epsilon_transitions()
det = nfa_eps.determinize()
print('déterministe ?          :', nfa_eps.is_deterministic())
print('états après déterminisation :', len(det.states))
for w in ['b', 'aab', 'ab', 'aa']:
    print(repr(w).ljust(6), nfa_eps.accepts_input(w), sans_eps.accepts_input(w), det.accepts_input(w))

déterministe ?          : False
états après déterminisation : 3
'b'    True True True
'aab'  True True True
'ab'   True True True
'aa'   False False False


## 8. Grammaire régulière associée

In [9]:
print(dfa.trim().to_grammar())

1 -> a2
2 -> a2 | ε


## 9. Export Graphviz (DOT)

In [10]:
dot = dfa.export_dot()
print('\n'.join(dot.splitlines()[:8]))
print('...  (', len(dot.splitlines()), 'lignes au total )')

digraph "" {
	graph [bb="0,0,241.81,117",
		rankdir=LR,
		ranksep=0.5
	];
	node [label="\N"];
	"58e24a11-974d-410f-aa8e-baf081eb4f2a"	[fontsize=14.0,
		height=0.05,
...  ( 58 lignes au total )


## 10. Vérification : `minimize` préserve le langage

Comparaison mot par mot sur tous les mots jusqu'à la longueur 6.

In [11]:
from itertools import product
mots = [''.join(p) for n in range(7) for p in product('ab', repeat=n)]
ok = all(redondant.accepts_input(w) == m1.accepts_input(w) == m2.accepts_input(w) for w in mots)
print(len(mots), 'mots testés, langage préservé :', ok)

127 mots testés, langage préservé : True


## 11. Antimirov contre Brzozowski

La dérivée d'Antimirov est un *ensemble* d'expressions : on obtient un NFA d'au plus n + 1 états. Pour « la 3e lettre avant la fin est un a », le DFA de Brzozowski explose (2³ = 8 états) alors que le NFA d'Antimirov en a 4.

In [12]:
A = {"a", "b"}
print("regex".ljust(20), "Brzozowski".ljust(11), "Antimirov".ljust(10), "Thompson")
for r in ["a*b", "(a|b)*abb", "(ab|a)*", "(a|b)*a(a|b)(a|b)", "ab*a|b"]:
    br = ExtendedDFA.brzozowski_automaton(r, A)
    an = ExtendedNFA.antimirov_automaton(r, A)
    th = ExtendedNFA.regex_to_nfa(r, input_symbols=A)
    ok = all(an.accepts_input(w) == th.accepts_input(w) for w in mots)
    print(r.ljust(20), str(len(br.states)).ljust(11), str(len(an.states)).ljust(10),
          len(th.states), "| même langage que Thompson :", ok)

regex                Brzozowski  Antimirov  Thompson
a*b                  3           2          5 | même langage que Thompson : True
(a|b)*abb            4           4          12 | même langage que Thompson : True
(ab|a)*              3           2          8 | même langage que Thompson : True
(a|b)*a(a|b)(a|b)    8           4          18 | même langage que Thompson : True
ab*a|b               4           3          10 | même langage que Thompson : True


## 12. ExtendedGNFA : l'ordre d'élimination des états

Le langage est le même quel que soit l'ordre, mais pas l'écriture de la regex.

In [13]:
g = d2.to_gnfa()
print("états éliminables :", g.eliminable_states())
print("ordre i, p :", g.to_regex_with_order(["i", "p"]))
print("ordre p, i :", g.to_regex_with_order(["p", "i"]))
print("natif      :", g.to_regex())
for k, step in enumerate(g.elimination_steps(["i", "p"])):
    print("étape", k, ": états restants =", len(step.states))

états éliminables : ['i', 'p']
ordre i, p : (ab*a|b)*
ordre p, i : b*a(ab*a|b)*ab*|b*
natif      : (ab*a|b)*
étape 0 : états restants = 4
étape 1 : états restants = 3
étape 2 : états restants = 2
